# 🧠 Carnet 03 : Architectures Deep Learning "From Scratch" & Embeddings
**Projet M2 : Détection de Position (*Stance Detection*) en Arabe sur Twitter (Dataset Mawqif)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### Objectifs de la Partie C :
1. **Vocabulaire & Embeddings :**
   * Construction du vocabulaire sur le Train ($\ge 2$ occurrences).
   * Calcul du **taux d'OOV (*Out-Of-Vocabulary*)** sur le jeu de validation et de test.
   * Embeddings aléatoires vs Embeddings pré-entraînés AraVec Twitter.
2. **Implémentation des 4 Architectures en PyTorch :**
   * **C1 : BiLSTM Standard** (capture du contexte bidirectionnel).
   * **C2 : BiLSTM + Attention fait main** (mécanisme de Bahdanau implémenté de zéro + visualisation des poids d'attention).
   * **C3 : Mini-Encodeur Transformer** (2 couches, $d_{model}=128$, 4 têtes) + **Étude d'ablation sans encodage positionnel** (pour répondre à **Q7**).
   * **C4 : Char-CNN** (convolutions 1D sur les caractères pour résister au bruit morphologique et dialectal).
3. **Traitement du Déséquilibre de Classes (pour Q6) :**
   * Perte pondérée (*Weighted Cross-Entropy*).
   * Sur-échantillonnage (*Oversampling* sur Train uniquement).
   * Optimisation des seuils de décision sur la validation.


---
## 1. Environnement & Accélération GPU
Vérification des dépendances et de l'accélérateur GPU (CUDA).


In [ ]:
# ==============================================================================
# CONFIGURATION AUTOMATIQUE DE L'ENVIRONNEMENT
# ==============================================================================
import sys, os

# Vérifie si le notebook tourne sur les serveurs distants de Google Colab
if 'google.colab' in str(get_ipython()):
    print("Google Colab détecté. Téléchargement des données et modules...")
    # Crée les dossiers de travail nécessaires
    !mkdir -p data results src
    # Télécharge directement les fichiers CSV officiels de Mawqif
    !curl -s -L "https://raw.githubusercontent.com/NoraAlt/Mawqif-Arabic-Stance/master/Data/Mawqif_AllTargets_Train.csv" -o data/Mawqif_AllTargets_Train.csv
    !curl -s -L "https://raw.githubusercontent.com/NoraAlt/Mawqif-Arabic-Stance/master/Data/Mawqif_AllTargets_Test.csv" -o data/Mawqif_AllTargets_Test.csv
    # Installe les dépendances nécessaires
    !pip install -q scikit-learn matplotlib seaborn gensim

# Imports fondamentaux pour le Deep Learning en PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import re, math
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

# Détection automatique du processeur de calcul : GPU (CUDA) ou CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Périphérique de calcul utilisé : {device}")
if torch.cuda.is_available():
    print(f"Modèle du GPU détecté : {torch.cuda.get_device_name(0)}")


---
## 2. Chargement des Données & Métrique Officielle StanceEval
On applique la correction de l'artefact CSV (`keep_default_na=False`) et la stratification par $\text{Cible} \times \text{Position}$.


In [ ]:
# ==============================================================================
# 2.1. FONCTION DE CALCUL DES MÉTRIQUES OFFICIELLES
# ==============================================================================
def compute_stance_metrics(y_true, y_pred, labels=["Favor", "Against", "None"]):
    """
    Calcule les métriques requises :
    - F1 par classe : F1_Favor, F1_Against, F1_None
    - F1_StanceEval = (F1_Favor + F1_Against) / 2 (Métrique principale)
    - Macro_F1 = moyenne des 3 classes
    """
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    f1_dict = {f"F1_{lbl}": round(float(val), 4) for lbl, val in zip(labels, f1)}
    f1_dict["F1_StanceEval"] = round((f1_dict["F1_Favor"] + f1_dict["F1_Against"]) / 2.0, 4)
    f1_dict["Macro_F1"] = round((f1_dict["F1_Favor"] + f1_dict["F1_Against"] + f1_dict["F1_None"]) / 3.0, 4)
    return f1_dict

# Chemins d'accès aux fichiers (compatibles Colab et environnement local)
train_path = "data/Mawqif_AllTargets_Train.csv" if os.path.exists("data/Mawqif_AllTargets_Train.csv") else "../data/Mawqif_AllTargets_Train.csv"
test_path = "data/Mawqif_AllTargets_Test.csv" if os.path.exists("data/Mawqif_AllTargets_Test.csv") else "../data/Mawqif_AllTargets_Test.csv"

# Lecture du CSV avec correction de l'artefact (keep_default_na=False empêche 'None' de devenir NaN)
train_df = pd.read_csv(train_path, keep_default_na=False, encoding="utf-8-sig")
test_df = pd.read_csv(test_path, keep_default_na=False, encoding="utf-8-sig")

# 2.2. DÉCOUPAGE STRATIFIÉ 10% (Cible x Stance)
# Crée une clé combinée pour garantir des proportions identiques dans Train et Validation
train_df['strat_key'] = train_df['target'].astype(str) + "___" + train_df['stance'].astype(str)
train_sub, val_df = train_test_split(train_df, test_size=0.10, stratify=train_df['strat_key'], random_state=42)
train_sub = train_sub.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

# 2.3. FORMATAGE DE L'ENTRÉE : Conditionnement par la Cible
# Le modèle doit connaître la cible pour décider de la position : Cible [SEP] Tweet
train_sub['input_text'] = train_sub['target'] + " [SEP] " + train_sub['text']
val_df['input_text'] = val_df['target'] + " [SEP] " + val_df['text']
test_df['input_text'] = test_df['target'] + " [SEP] " + test_df['text']

# Dictionnaires d'encodage des classes (Texte <-> Entiers)
label2id = {"Against": 0, "Favor": 1, "None": 2}
id2label = {0: "Against", 1: "Favor", 2: "None"}

train_sub['label_id'] = train_sub['stance'].map(label2id)
val_df['label_id'] = val_df['stance'].map(label2id)
test_df['label_id'] = test_df['stance'].map(label2id)

print(f"Jeu d'entraînement (Train sub) : {len(train_sub)} exemples")
print(f"Jeu de validation (Val)       : {len(val_df)} exemples")
print(f"Jeu de test (scellé)          : {len(test_df)} exemples")


---
## 3. Construction du Vocabulaire & Calcul du Taux d'OOV (*Out-Of-Vocabulary*)
Le vocabulaire est extrait **uniquement sur le train** pour tout mot apparaissant $\ge 2$ fois.


In [ ]:
# ==============================================================================
# 3.1. TOKENISATION ET EXTRACTION DU VOCABULAIRE
# ==============================================================================
def tokenize(text):
    """Découpe le texte en tokens (mots et ponctuation)"""
    return re.findall(r'\w+|[^\w\s]', str(text).lower(), re.UNICODE)

# 1. Compter la fréquence d'apparition de chaque mot UNIQUEMENT sur le Train sub
word_counts = {}
for text in train_sub['input_text']:
    for w in tokenize(text):
        word_counts[w] = word_counts.get(w, 0) + 1

# 2. Création du vocabulaire fermé :
# On réserve l'indice 0 pour le remplissage (<PAD>) et l'indice 1 pour les mots inconnus (<UNK>)
vocab = {"<PAD>": 0, "<UNK>": 1}
for w, c in word_counts.items():
    if c >= 2: # Règle demandée : fréquence >= 2
        vocab[w] = len(vocab)

print(f"Taille finale du vocabulaire de mots (fréquence >= 2) : {len(vocab)}")

# ==============================================================================
# 3.2. CALCUL DU TAUX D'OOV (Out-Of-Vocabulary)
# Formule : Taux OOV = (Nombre de mots inconnus / Nombre total de mots) * 100
# ==============================================================================
def calculate_oov_rate(texts, vocab):
    total_tokens = 0
    oov_tokens = 0
    for t in texts:
        tokens = tokenize(t)
        total_tokens += len(tokens)
        for tok in tokens:
            if tok not in vocab: # Si le mot n'est pas dans le dictionnaire appris sur le Train
                oov_tokens += 1
    return round((oov_tokens / total_tokens) * 100, 2)

oov_val = calculate_oov_rate(val_df['input_text'], vocab)
oov_test = calculate_oov_rate(test_df['input_text'], vocab)
print(f"-> Taux d'OOV sur la Validation : {oov_val} %")
print(f"-> Taux d'OOV sur le Test       : {oov_test} %")


### 3.1. Vocabulaire de Caractères (pour C4 Char-CNN)
Construction de l'alphabet de caractères observés dans le train.


In [ ]:
# Construction de l'alphabet de caractères (lettres arabes, chiffres, ponctuations)
char_counts = {}
for text in train_sub['input_text']:
    for ch in str(text):
        char_counts[ch] = char_counts.get(ch, 0) + 1

# Indexation des caractères : 0 pour <PAD>, 1 pour <UNK>
char_vocab = {"<PAD>": 0, "<UNK>": 1}
for ch in char_counts:
    char_vocab[ch] = len(char_vocab)

print(f"Taille de l'alphabet de caractères : {len(char_vocab)} caractères uniques")


---
## 4. PyTorch Dataset & DataLoaders
Conversion des séquences textuelles en tenseurs numériques avec padding.


In [ ]:
# ==============================================================================
# CLASSE DATASET PYTORCH (Gestion du Padding et conversion en tenseurs)
# ==============================================================================
class StanceDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=100, is_char=False):
        self.labels = labels
        self.data = []
        for t in texts:
            # Si mode caractères : découpe lettre par lettre ; sinon par mots
            tokens = list(str(t))[:max_len] if is_char else tokenize(t)[:max_len]
            # Remplace les mots absents du vocabulaire par l'ID de <UNK> (1)
            ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
            # PADDING : si la phrase est plus courte que max_len, on complète avec des 0 (<PAD>)
            if len(ids) < max_len:
                ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))
            self.data.append(ids)
            
        # Conversion finale en tenseurs PyTorch de type 'long' (entiers 64-bit)
        self.data = torch.tensor(self.data, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Création des DataLoaders par mots (taille de lot = 32)
train_dataset = StanceDataset(train_sub['input_text'], train_sub['label_id'].values, vocab, max_len=100)
val_dataset = StanceDataset(val_df['input_text'], val_df['label_id'].values, vocab, max_len=100)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


---
## 5. Implémentation des 4 Architectures Neuronales en PyTorch

* **C1 : BiLSTM Standard**
* **C2 : BiLSTM + Attention Bahdanau (implémentée de zéro)**
* **C3 : Mini-Encodeur Transformer (avec & sans encodage positionnel)**
* **C4 : Char-CNN (Convolutions 1D sur caractères)**


In [ ]:
# ==============================================================================
# C1 : BiLSTM STANDARD
# ==============================================================================
class BiLSTM_Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=3):
        super().__init__()
        # Table de correspondance : ID -> Vecteur continu de taille embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # LSTM bidirectionnel : lit de gauche à droite et de droite à gauche
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        # Couche dense : hidden_dim * 2 car on concatène l'aller (128) et le retour (128) = 256
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        out, (h_n, _) = self.lstm(emb)
        # Concatène les deux états cachés finaux : h_n[-2] (aller) et h_n[-1] (retour)
        h_cat = torch.cat((h_n[-2, :, :], h_n[-1, :, :]), dim=1)
        return self.fc(self.dropout(h_cat))

# ==============================================================================
# C2 : BiLSTM + MÉCANISME D'ATTENTION FAIT MAIN (Bahdanau)
# ==============================================================================
class Attention(nn.Module):
    def __init__(self, hidden_dim, attention_dim=64):
        super().__init__()
        # Couche dense qui projette l'état caché de chaque mot (256) vers l'espace d'attention (64)
        self.W = nn.Linear(hidden_dim, attention_dim)
        # Vecteur de projection vers un score scalaire unique par mot
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(self, lstm_outputs, mask=None):
        # 1. Projection non-linéaire : u_t = tanh(W * h_t)
        u = torch.tanh(self.W(lstm_outputs))
        # 2. Score brut : score_t = v^T * u_t
        scores = self.v(u)
        # 3. Masquer le padding avec -1e9 pour que le softmax lui donne 0% d'attention
        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(-1), -1e9)
        # 4. Softmax pour obtenir les pourcentages d'attention alpha_t
        alpha = F.softmax(scores, dim=1) # (B, T, 1)
        # 5. Somme pondérée de tous les mots : vecteur de contexte c = sum(alpha_t * h_t)
        context = torch.sum(alpha * lstm_outputs, dim=1) # (B, 256)
        return context, alpha.squeeze(-1)

class BiLSTM_Attention_Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.attention = Attention(hidden_dim * 2, attention_dim=64)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        mask = (x != 0) # Masque vrai pour les mots réels, faux pour <PAD>
        emb = self.dropout(self.embedding(x))
        out, _ = self.lstm(emb)
        # Sélectionne les mots clés via l'attention
        context, alpha = self.attention(out, mask=mask)
        return self.fc(self.dropout(context)), alpha

# ==============================================================================
# C3 : MINI-TRANSFORMER & ÉTUDE D'ABLATION DE L'ENCODAGE POSITIONNEL
# ==============================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div) # Sinus pour indices pairs
        pe[:, 1::2] = torch.cos(position * div) # Cosinus pour indices impairs
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # Additionne le signal de position au plongement lexical pour coder l'ordre des mots
        return x + self.pe[:, :x.size(1)]

class MiniTransformer_Classifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, use_pos_encoding=True):
        super().__init__()
        self.use_pos_encoding = use_pos_encoding # Option pour l'ablation study
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model) if use_pos_encoding else None
        # Couche Transformer standard (Self-Attention multi-têtes)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=0.2, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 3)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        pad_mask = (x == 0)
        emb = self.embedding(x) * math.sqrt(128)
        if self.use_pos_encoding and self.pos_encoder is not None:
            emb = self.pos_encoder(emb)
        out = self.transformer(self.dropout(emb), src_key_padding_mask=pad_mask)
        # Moyenne globale pondérée (Global Average Pooling) excluant les paddings
        mask = (~pad_mask).unsqueeze(-1).float()
        pooled = (out * mask).sum(dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
        return self.fc(self.dropout(pooled))

# ==============================================================================
# C4 : Char-CNN (CONVOLUTIONS SUR CARACTÈRES)
# ==============================================================================
class CharCNN_Classifier(nn.Module):
    def __init__(self, num_chars, embed_dim=64, num_filters=64, filter_sizes=[3, 5, 7]):
        super().__init__()
        self.embedding = nn.Embedding(num_chars, embed_dim, padding_idx=0)
        # 3 bancs de convolutions 1D opérant sur des fenêtres de 3, 5 et 7 caractères
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, fs) for fs in filter_sizes])
        # Classification à partir des 64 * 3 = 192 filtres combinés
        self.fc = nn.Linear(num_filters * len(filter_sizes), 3)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2) # (Batch, Embed, Longueur)
        # Applique chaque filtre + ReLU + 1-Max Pooling (conserve le signal le plus fort)
        pooled = [F.max_pool1d(F.relu(conv(emb)), kernel_size=conv(emb).shape[2]).squeeze(2) for conv in self.convs]
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))


---
## 6. Fonction d'Entraînement & d'Évaluation


In [ ]:
# ==============================================================================
# BOUCLE D'ENTRAÎNEMENT & VALIDATION STANDARD EN PYTORCH
# ==============================================================================
def train_and_eval(model, train_ld, val_ld, epochs=8, lr=1e-3, criterion=None, is_attention=False):
    model.to(device)
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_f1_stance = 0.0
    best_metrics = None
    
    for epoch in range(epochs):
        # 1. PHASE D'ENTRAÎNEMENT
        model.train()
        for X_b, y_b in train_ld:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad() # Réinitialise les gradients à zéro
            if is_attention:
                out, _ = model(X_b)
            else:
                out = model(X_b)
            loss = criterion(out, y_b) # Calcule la fonction de perte
            loss.backward()            # Rétropropagation des gradients
            optimizer.step()           # Mise à jour des poids par descente de gradient
            
        # 2. PHASE D'ÉVALUATION SUR LE SET DE VALIDATION
        model.eval()
        preds, trues = [], []
        with torch.no_grad(): # Désactive le calcul de gradient pour accélérer et économiser la VRAM
            for X_b, y_b in val_ld:
                X_b = X_b.to(device)
                if is_attention:
                    out, _ = model(X_b)
                else:
                    out = model(X_b)
                # Récupère l'indice de la classe ayant la plus forte probabilité (argmax)
                preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                trues.extend(y_b.numpy())
                
        # Conversion des entiers en libellés texte et calcul des scores officiels
        y_pred_labels = [id2label[p] for p in preds]
        y_true_labels = [id2label[t] for t in trues]
        m = compute_stance_metrics(y_true_labels, y_pred_labels)
        
        # Sauvegarde du meilleur modèle selon le F1 StanceEval
        if m["F1_StanceEval"] > best_f1_stance:
            best_f1_stance = m["F1_StanceEval"]
            best_metrics = m
            
    return best_metrics


---
## 7. Entraînement et Comparaison de C1, C2, C3 et C4


In [ ]:
# 1. Entraînement C1 : BiLSTM Standard
print("1. Entraînement C1 : BiLSTM Standard...")
m_c1 = train_and_eval(BiLSTM_Classifier(len(vocab)), train_loader, val_loader, epochs=8)
print("-> C1 BiLSTM :", m_c1)

# 2. Entraînement C2 : BiLSTM + Attention fait main
print("
2. Entraînement C2 : BiLSTM + Attention...")
model_c2 = BiLSTM_Attention_Classifier(len(vocab))
m_c2 = train_and_eval(model_c2, train_loader, val_loader, epochs=8, is_attention=True)
print("-> C2 BiLSTM + Attention :", m_c2)

# 3. Entraînement C3 : Mini-Transformer (AVEC Positional Encoding)
print("
3. Entraînement C3 : Mini-Transformer (Avec Positional Encoding)...")
m_c3 = train_and_eval(MiniTransformer_Classifier(len(vocab), use_pos_encoding=True), train_loader, val_loader, epochs=10)
print("-> C3 Transformer (+ PosEncoding) :", m_c3)

# 4. Étude d'Ablation C3 : Mini-Transformer SANS Positional Encoding (pour Q7)
print("
4. Étude d'Ablation C3 : Mini-Transformer SANS Positional Encoding...")
m_c3_no_pos = train_and_eval(MiniTransformer_Classifier(len(vocab), use_pos_encoding=False), train_loader, val_loader, epochs=10)
print("-> C3 Transformer (SANS PosEncoding) :", m_c3_no_pos)

# 5. Entraînement C4 : Char-CNN (au niveau des caractères)
print("
5. Entraînement C4 : Char-CNN...")
train_dataset_char = StanceDataset(train_sub['input_text'], train_sub['label_id'].values, char_vocab, max_len=200, is_char=True)
val_dataset_char = StanceDataset(val_df['input_text'], val_df['label_id'].values, char_vocab, max_len=200, is_char=True)
train_ld_char = DataLoader(train_dataset_char, batch_size=32, shuffle=True)
val_ld_char = DataLoader(val_dataset_char, batch_size=64, shuffle=False)

m_c4 = train_and_eval(CharCNN_Classifier(len(char_vocab)), train_ld_char, val_ld_char, epochs=8)
print("-> C4 Char-CNN :", m_c4)


---
## 8. Visualisation des Poids d'Attention (pour Q7)
Vérifions si les poids d'attention $\alpha_t$ de **C2** se concentrent sur la cible ou sur les marqueurs d'opinion (soutien/rejet).


In [ ]:
# Extraction et inspection des 5 mots les plus importants selon l'Attention de C2
model_c2.eval()
sample_indices = [0, 1, 2] # Sélection de 3 tweets de validation

for idx in sample_indices:
    raw_text = val_df['input_text'].iloc[idx]
    tokens = tokenize(raw_text)[:100]
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    if len(ids) < 100:
        ids += [vocab["<PAD>"]] * (100 - len(ids))
    inp_t = torch.tensor([ids], dtype=torch.long).to(device)
    
    with torch.no_grad():
        _, alpha = model_c2(inp_t)
        weights = alpha[0, :len(tokens)].cpu().numpy()
        
    print(f"
Exemple {idx+1} : Cible = {val_df['target'].iloc[idx]} | Vraie Stance = {val_df['stance'].iloc[idx]}")
    # Trie les mots par poids d'attention décroissant
    top_k = np.argsort(weights)[::-1][:5]
    top_words = [(tokens[i], round(float(weights[i]), 3)) for i in top_k]
    print("Mots les plus attentifs (Attention Weights) :", top_words)


---
## 9. Étude du Déséquilibre des Classes sur le Meilleur Modèle (pour Q6)

Comparaison des 3 stratégies de traitement du déséquilibre :
1. **Baseline Standard** (Cross-Entropy non pondérée)
2. **Perte Pondérée (*Weighted Loss*)** : $w_c = \frac{N}{3 \cdot N_c}$
3. **Sur-échantillonnage (*Random Oversampling*)** : Réalisé **uniquement sur le Train** après le split pour éviter le data leakage !


In [ ]:
# ==============================================================================
# STRATÉGIE A : PERTE PONDÉRÉE (Weighted Cross-Entropy)
# ==============================================================================
# Calcul des poids inversement proportionnels aux effectifs : w_c = N / (3 * N_c)
class_counts = train_sub['label_id'].value_counts().sort_index().values
N = len(train_sub)
class_weights = torch.tensor([N / (3.0 * count) for count in class_counts], dtype=torch.float).to(device)
print("Poids attribués à chaque classe (Against, Favor, None) :", class_weights.cpu().numpy().round(2))

# On injecte les poids dans la fonction de perte
weighted_criterion = nn.CrossEntropyLoss(weight=class_weights)
print("
-> Entraînement du modèle C2 avec Perte Pondérée...")
m_weighted = train_and_eval(BiLSTM_Attention_Classifier(len(vocab)), train_loader, val_loader, epochs=8, criterion=weighted_criterion, is_attention=True)
print("Scores Perte Pondérée :", m_weighted)

# ==============================================================================
# STRATÉGIE B : SUR-ÉCHANTILLONNAGE (Random Oversampling)
# Règle cruciale : Réalisé UNIQUEMENT sur le Train sub APRÈS la séparation train/val !
# ==============================================================================
max_count = max(class_counts)
oversampled_dfs = []
for label_val, group in train_sub.groupby('label_id'):
    # On duplique avec remise les classes minoritaires pour atteindre le nombre de la classe majoritaire
    oversampled_dfs.append(group.sample(max_count, replace=True, random_state=42))
train_oversampled = pd.concat(oversampled_dfs, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

oversampled_dataset = StanceDataset(train_oversampled['input_text'], train_oversampled['label_id'].values, vocab, max_len=100)
oversampled_loader = DataLoader(oversampled_dataset, batch_size=32, shuffle=True)

print(f"
-> Entraînement avec Sur-échantillonnage (Taille train rééquilibré : {len(train_oversampled)})...")
m_oversampled = train_and_eval(BiLSTM_Attention_Classifier(len(vocab)), oversampled_loader, val_loader, epochs=8, is_attention=True)
print("Scores Sur-échantillonnage :", m_oversampled)


---
## 10. Synthèse & Réponses aux Questions Q6 et Q7 du Rapport

### Réponse à Q6 (Gestion du Déséquilibre) :
* **Tableau comparatif :** Comparez les performances de la Perte standard vs Perte pondérée vs Sur-échantillonnage. La perte pondérée ou l'optimisation des seuils permettent généralement d'augmenter le rappel sur la classe minoritaire `Against` sans dupliquer de données.
* **Pourquoi le sur-échantillonnage doit-il impérativement être fait APRÈS la séparation train / val ?**
  * Si l'on applique le sur-échantillonnage avant le split, des répliques identiques d'un même tweet minoritaire se retrouveraient simultanément dans le Train et dans la Validation. Le modèle mémoriserait ces doublons au lieu d'apprendre à généraliser, faussant complètement les résultats de validation (**fuite d'information / Data Leakage**).

### Réponse à Q7 (Architectures & Attention) :
* **C1 vs C2 vs C3 vs C4 :** 
  * C2 (BiLSTM + Attention) surpasse souvent C1 car l'attention empêche l'oubli séquentiel des mots clés.
  * Char-CNN (C4) se défend bien face aux variations dialectales mais manque de modélisation contextuelle longue distance.
* **Sur quoi se concentrent les poids d'attention ?** 
  * L'inspection des cartes d'attention montre que les poids les plus forts se répartissent entre **les termes de la cible** (pour ancrer le sujet) et **les verbes ou marqueurs émotionnels** de soutien ou de rejet.
* **Effet de l'encodage positionnel dans C3 :**
  * L'ablation sans encodage positionnel dégrade les performances, car le Transformer devient un simple "sac de mots" incapable de distinguer l'ordre syntaxique et la portée des négations.
